# Valkey Search Workshop — Streaming Service Demo

## 3-Hour Hands-On Workshop

In this workshop you'll build a movie recommendation system using **Valkey Search**, learning:

1. **Index creation** — TEXT, TAG, NUMERIC, VECTOR fields
2. **FT.SEARCH** — Full-text search, tag filters, numeric ranges
3. **Vector similarity (KNN)** — Find similar movies using embeddings
4. **Hybrid search** — Combine filters with vector similarity
5. **Single-slot indexes** — Microsecond per-user queries
6. **FT.AGGREGATE** — Platform analytics (trending, most watched)
7. **Cross-index workflows** — Connecting user history to catalog recommendations

### Architecture

| Index | Purpose | Latency |
|:------|:--------|:--------|
| `idx:movies` (Global) | Movie catalog with vectors | Low ms |
| `idx:user:<id>:history` (Single-Slot) | One user's watch history | Sub-ms |
| `idx:watch` (Global) | All users' history for analytics | Tens of ms |

### Dataset
- **1,053 movies** with 768-dim embeddings (from TMDB)
- **1,958 ratings** from 20 users (from MovieLens)


## Part 1: Setup & Connection (~15 min)

### Option A: Google Colab
Run the cell below to install Valkey with the Search module directly in Colab.

### Option B: Local Machine
Run `docker compose up -d` in the workshop directory, then skip the install cell.


In [1]:
# === SETUP ===
# Starts Valkey with Search module using Docker or Podman.
# Works on Windows (Podman/Docker) and Mac (Podman/Colima/Docker).

import subprocess, socket, time, shutil

# Detect container runtime
runtime = 'podman' if shutil.which('podman') else 'docker'
print(f'Using: {runtime}')

# Stop and remove any existing Valkey container
subprocess.run([runtime, 'rm', '-f', 'valkey'], capture_output=True)
time.sleep(1)

# Start Valkey
subprocess.run([runtime, 'run', '-d', '--name', 'valkey', '-p', '6379:6379',
    'valkey/valkey-bundle:9.1.0-rc2',
    'valkey-server', '--save', '', '--protected-mode', 'no'])

print('Waiting for Valkey to start...')
time.sleep(5)

# Verify by pinging inside the container (bypasses Windows networking issues)
result = subprocess.run([runtime, 'exec', 'valkey', 'valkey-cli', 'ping'],
    capture_output=True, text=True)
assert 'PONG' in result.stdout, f'Failed to start. Check: {runtime} logs valkey'
print('✅ Valkey started')


Using: podman
92782f4d82270dfbbbc6c8cceac66f33c4c3da80199fe93c251e302b05fd6ea7
Waiting for Valkey to start...
✅ Valkey started


> **Prerequisites**: Docker installed and running.  
> If the cell above fails, run manually:  
> ```
> docker run -d --name valkey -p 6379:6379 valkey/valkey-bundle:9.1.0-rc2 valkey-server --save "" --protected-mode no
> ```

In [2]:
%pip install -q valkey pandas numpy

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
required = ['catalog.csv', 'movies.csv', 'ratings.csv', 'users.txt']
missing = [f for f in required if not os.path.exists(f'data/{f}')]
if missing:
    raise FileNotFoundError(f'Missing data files: {missing}. Upload the data/ folder.')
print(f'✅ Data ready: {os.listdir("data")}')

✅ Data ready: ['links.csv', 'ratings.csv', 'users.txt', 'catalog.csv', 'movies.csv']


In [4]:
import csv, struct, time, subprocess, json
import pandas as pd
import numpy as np
import valkey

# Connect to Valkey — handle Podman Windows networking
VALKEY_HOST = 'localhost'
VALKEY_PORT = 6379

try:
    r = valkey.Valkey(host=VALKEY_HOST, port=VALKEY_PORT, decode_responses=True, socket_timeout=3)
    r.ping()
except Exception:
    # Podman on Windows: localhost may not route. Get container IP instead.
    inspect = subprocess.run([runtime, 'inspect', 'valkey'], capture_output=True, text=True)
    info = json.loads(inspect.stdout)
    VALKEY_HOST = info[0]['NetworkSettings']['IPAddress'] or 'localhost'
    print(f'localhost failed, using container IP: {VALKEY_HOST}')
    r = valkey.Valkey(host=VALKEY_HOST, port=VALKEY_PORT, decode_responses=True, socket_timeout=3)

r_bin = valkey.Valkey(host=VALKEY_HOST, port=VALKEY_PORT, decode_responses=False, socket_timeout=3)

print(f'Connected: {r.ping()}')
modules = r_bin.module_list()
module_names = [m[b'name'].decode() for m in modules]
print(f'Modules: {module_names}')
assert 'search' in module_names, 'ERROR: search module not loaded!'
print('✅ Ready!')


Connected: True
Modules: ['bf', 'ldap', 'search', 'json', 'lua']
✅ Ready!


In [5]:
# Explore the datasets
import pandas as pd

# --- Data Sources ---
# Two original sources:
#   TMDB (via HuggingFace) → overview, vote_average, popularity, language, 768-dim vectors
#   MovieLens (ml-32m)     → user ratings, movie titles, genres, movieId↔tmdbId mapping
#
# --- Workshop CSVs ---
#
# catalog.csv  | Source: TMDB (HuggingFace parquet)
#              | Created: 1,053 rows extracted from 851K TMDB movies, matching tmdbIds
#              |          of movies our 20 users watched. Vectors are 768-dim PCA on
#              |          TMDB text embeddings (title+genres+overview).
#              | Used for: global catalog (title, overview, genres, vector)
#              | Index: idx:movies
#
# ratings.csv  | Source: MovieLens 32M
#              | Created: subset of ml-32m/ratings.csv — 20 users with 50-200 ratings
#              |          each (1,958 rows out of 32M).
#              | Used for: user ratings, joined with movies.csv for both indexes below
#              | Index: idx:user:<id>:history + idx:watch
#
# movies.csv   | Source: MovieLens 32M + ml-32m/links.csv
#              | Created: subset of ml-32m/movies.csv merged with links.csv to add
#              |          tmdbId. Only the 1,053 movies our 20 users rated.
#              | Used for: provides title, genres, tmdbId for each rating
#              | Index: idx:user:<id>:history + idx:watch
#
# users.txt    | 20 user IDs picked from ratings.csv

print('=== catalog.csv (from TMDB — global catalog) ===')
catalog = pd.read_csv('data/catalog.csv', encoding='utf-8', nrows=3)
print(f'Rows: {sum(1 for _ in open("data/catalog.csv", encoding="utf-8")) - 1} movies')
print(f'Columns: {list(catalog.columns)}')
print(catalog.drop(columns=['vector', 'overview']).to_string(index=False))

print(f'\n=== ratings.csv (from MovieLens — user ratings) ===')
ratings = pd.read_csv('data/ratings.csv', encoding='utf-8')
print(f'Rows: {len(ratings)} ratings | Users: {ratings["userId"].nunique()} | Movies: {ratings["movieId"].nunique()}')
print(f'Rating range: {ratings["rating"].min()} – {ratings["rating"].max()}')
print(ratings.head(3).to_string(index=False))

print(f'\n=== movies.csv (from MovieLens — joins ratings to catalog via tmdbId) ===')
movies = pd.read_csv('data/movies.csv', encoding='utf-8')
print(f'Rows: {len(movies)} | Columns: {list(movies.columns)}')
print(movies.head(3).to_string(index=False))


=== catalog.csv (movie catalog with vectors) ===
Rows: 1053 movies
Columns: ['id', 'title', 'overview', 'genres', 'vote_average', 'popularity', 'original_language', 'vector']
 id        title                             genres  vote_average  popularity original_language
  5   Four Rooms                             Comedy         5.900      24.557                en
 11    Star Wars Adventure, Action, Science Fiction         8.203     101.129                en
 12 Finding Nemo                  Animation, Family         7.800      83.434                en

=== ratings.csv (user watch history) ===
Rows: 1958 ratings
Users: 20 | Movies rated: 1053
Rating range: 0.5 – 5.0
 userId  movieId  rating  timestamp
      1       17     4.0  944249077
      1       25     1.0  944250228
      1       29     2.0  943230976

=== users.txt (demo users) ===
20 user IDs: ['1', '2', '3', '9', '13', '15', '17', '18', '20', '22', '23', '25', '27', '29', '31', '34', '36', '39', '40', '43']
Each user has ~97 r

## Part 2: Global Catalog Index (~45 min)

A **global index** is distributed across all shards in a cluster.
- Each shard holds a portion of the documents
- Queries fan out to every shard, results are merged and returned
- Supports TEXT, TAG, NUMERIC, and VECTOR fields

We load documents first (plain HSET), then create the index. Valkey backfills existing keys asynchronously.

### 2.1 Load Data


In [6]:
# Example: what one document looks like in the catalog
# Each movie becomes a HASH key: movie:<tmdb_id>
print('''
Key:    movie:11
Fields:
  title             = "Star Wars"                    (TEXT — full-text searchable)
  overview          = "Princess Leia is captured..." (TEXT — searchable description)
  genres            = "Adventure,Action,Sci-Fi"      (TAG — comma-separated, filterable)
  original_language = "en"                           (TAG — exact match filter)
  vote_average      = 8.2                            (NUMERIC — range queries, sortable)
  popularity        = 101.1                          (NUMERIC — range queries, sortable)
  vector            = <3072 bytes>                   (VECTOR — 768 × float32, for KNN)
''')



Key:    movie:11
Fields:
  title             = "Star Wars"                    (TEXT — full-text searchable)
  overview          = "Princess Leia is captured..." (TEXT — searchable description)
  genres            = "Adventure,Action,Sci-Fi"      (TAG — comma-separated, filterable)
  original_language = "en"                           (TAG — exact match filter)
  vote_average      = 8.2                            (NUMERIC — range queries, sortable)
  popularity        = 101.1                          (NUMERIC — range queries, sortable)
  vector            = <3072 bytes>                   (VECTOR — 768 × float32, for KNN)



In [7]:
# Start fresh
r.flushall()

# Load movies from CSV with binary-packed vectors
count = 0
pipe = r_bin.pipeline(transaction=False)
t0 = time.time()

with open('data/catalog.csv', 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        # Pack 768 floats into binary blob (required for vector indexing)
        vec_floats = [float(x) for x in row['vector'].split(',')]
        vec_blob = struct.pack(f'{len(vec_floats)}f', *vec_floats)
        
        pipe.hset(f'movie:{row["id"]}'.encode(), mapping={
            b'title': row['title'].encode(),
            b'overview': row['overview'].encode(),
            b'genres': row['genres'].encode(),
            b'original_language': row['original_language'].encode(),
            b'vote_average': row['vote_average'].encode(),
            b'popularity': row['popularity'].encode(),
            b'vector': vec_blob,
        })
        count += 1
        if count % 500 == 0:
            pipe.execute()
            pipe = r_bin.pipeline(transaction=False)

pipe.execute()
elapsed = time.time() - t0
print(f"✓ Loaded {count} movies in {elapsed:.1f}s ({count/elapsed:.0f} docs/sec)")

✓ Loaded 1053 movies in 0.2s (4219 docs/sec)


### 2.2 Create the Index

Now create the index — Valkey backfills existing keys asynchronously.


In [8]:
# Drop if exists
try:
    r.execute_command('FT.DROPINDEX', 'idx:movies')
except:
    pass

# Create the global catalog index
r.execute_command('FT.CREATE', 'idx:movies', 'ON', 'HASH', 'PREFIX', '1', 'movie:',
    'SCHEMA',
    'title', 'TEXT',                          # Full-text searchable
    'overview', 'TEXT',                       # Movie description
    'genres', 'TAG', 'SEPARATOR', ',',        # Filterable categories
    'original_language', 'TAG',               # Language filter
    'vote_average', 'NUMERIC', 'SORTABLE',    # Rating 0-10
    'popularity', 'NUMERIC', 'SORTABLE',      # Popularity score
    'vector', 'VECTOR', 'FLAT', '6',          # Vector similarity
        'TYPE', 'FLOAT32', 'DIM', '768', 'DISTANCE_METRIC', 'COSINE')

print("✓ Index idx:movies created")

✓ Index idx:movies created


In [9]:
# Wait for backfill to complete
import time
while True:
    info = r.execute_command('FT.INFO', 'idx:movies')
    info_dict = dict(zip(info[::2], info[1::2]))
    pct = float(info_dict.get('backfill_complete_percent', 1.0))
    docs = info_dict.get('num_docs', 0)
    if pct >= 1.0:
        break
    print(f'  Indexing... {pct*100:.0f}% ({docs} docs)', end='\r')
    time.sleep(1)
print(f'✓ Index ready: {info_dict.get("num_docs", 0)} docs indexed')

✓ Index ready: 1053 docs indexed


In [10]:
# Inspect the index
r.execute_command('FT.INFO', 'idx:movies')


['index_name',
 'idx:movies',
 'index_definition',
 ['key_type', 'HASH', 'prefixes', ['movie:'], 'default_score', '1'],
 'attributes',
 [['identifier',
   'original_language',
   'attribute',
   'original_language',
   'user_indexed_memory',
   2106,
   'type',
   'TAG',
   'SEPARATOR',
   ',',
   'CASESENSITIVE',
   '0',
   'size',
   '1053'],
  ['identifier',
   'vote_average',
   'attribute',
   'vote_average',
   'user_indexed_memory',
   3842,
   'type',
   'NUMERIC',
   'size',
   '1053'],
  ['identifier',
   'vector',
   'attribute',
   'vector',
   'user_indexed_memory',
   3234816,
   'type',
   'VECTOR',
   'index',
   ['capacity',
    10240,
    'dimensions',
    768,
    'distance_metric',
    'COSINE',
    'size',
    '1053',
    'data_type',
    'FLOAT32',
    'algorithm',
    ['name', 'FLAT', 'block_size', 1024]]],
  ['identifier',
   'genres',
   'attribute',
   'genres',
   'user_indexed_memory',
   22440,
   'type',
   'TAG',
   'SEPARATOR',
   ',',
   'CASESENSITIVE'

### 2.3 Full-Text Search (FT.SEARCH)

Search movie titles and overviews using natural language queries.


In [11]:
# Simple text search
results = r.execute_command('FT.SEARCH', 'idx:movies', 'space adventure',
    'RETURN', '2', 'title', 'genres',
    'LIMIT', '0', '5')

results  # Raw response: [total_matches, key1, [field, val, ...], key2, [...], ...]

[2,
 'movie:157336',
 ['title', 'Interstellar', 'genres', 'Adventure, Drama, Science Fiction'],
 'movie:10681',
 ['title', 'WALL·E', 'genres', 'Animation, Family, Science Fiction']]

In [12]:
# Search within a specific field
r.execute_command('FT.SEARCH', 'idx:movies', '@title:Matrix',
    'RETURN', '2', 'title', 'vote_average',
    'LIMIT', '0', '5')

[3,
 'movie:604',
 ['title', 'The Matrix Reloaded', 'vote_average', '7.054'],
 'movie:605',
 ['title', 'The Matrix Revolutions', 'vote_average', '6.731'],
 'movie:603',
 ['title', 'The Matrix', 'vote_average', '8.22']]

### 2.4 Tag & Numeric Filters

TAG fields support exact-match filtering. NUMERIC fields support range queries.


In [13]:
# Tag filter: exact match on genre
r.execute_command('FT.SEARCH', 'idx:movies', '@genres:{Action}',
    'RETURN', '2', 'title', 'genres',
    'LIMIT', '0', '5')

[269,
 'movie:7007',
 ['title', 'Rising Sun', 'genres', 'Action, Drama, Thriller'],
 'movie:2048',
 ['title', 'I, Robot', 'genres', 'Action, Science Fiction'],
 'movie:290859',
 ['title',
  'Terminator: Dark Fate',
  'genres',
  'Science Fiction, Action, Adventure'],
 'movie:12106',
 ['title', 'The Quick and the Dead', 'genres', 'Western, Action'],
 'movie:9383',
 ['title', 'Hollow Man', 'genres', 'Action, Science Fiction, Thriller']]

In [14]:
# Combined: tag + numeric range + sort
r.execute_command('FT.SEARCH', 'idx:movies',
    '@genres:{Action} @vote_average:[8 10]',
    'SORTBY', 'popularity', 'DESC',
    'RETURN', '3', 'title', 'vote_average', 'popularity',
    'LIMIT', '0', '5')

[15,
 'movie:98',
 ['title', 'Gladiator', 'vote_average', '8.2', 'popularity', '405.13'],
 'movie:155',
 ['title', 'The Dark Knight', 'vote_average', '8.5', 'popularity', '152.084'],
 'movie:120',
 ['title',
  'The Lord of the Rings: The Fellowship of the Ring',
  'vote_average',
  '8.4',
  'popularity',
  '144.458'],
 'movie:122',
 ['title',
  'The Lord of the Rings: The Return of the King',
  'vote_average',
  '8.483',
  'popularity',
  '141.342'],
 'movie:121',
 ['title',
  'The Lord of the Rings: The Two Towers',
  'vote_average',
  '8.4',
  'popularity',
  '124.714']]

In [15]:
# Multi-tag OR: Comedy | Drama, rated 8.5+
r.execute_command('FT.SEARCH', 'idx:movies',
    '@genres:{Comedy | Drama} @vote_average:[8.5 10]',
    'RETURN', '2', 'title', 'genres',
    'LIMIT', '0', '5')

[11,
 'movie:346',
 ['title', 'Seven Samurai', 'genres', 'Action, Drama'],
 'movie:13',
 ['title', 'Forrest Gump', 'genres', 'Comedy, Drama, Romance'],
 'movie:389',
 ['title', '12 Angry Men', 'genres', 'Drama'],
 'movie:155',
 ['title', 'The Dark Knight', 'genres', 'Drama, Action, Crime, Thriller'],
 'movie:372058',
 ['title', 'Your Name.', 'genres', 'Animation, Romance, Drama']]

### 2.5 Vector Similarity Search (KNN)

Each movie has a 768-dimensional embedding vector. We can find similar movies
using K-Nearest Neighbors (KNN) search with cosine distance.


In [16]:
# Vector KNN: find movies similar to Star Wars
source_vec = r_bin.hget(b'movie:11', b'vector')  # Star Wars
print(f"Source: {r.hget('movie:11', 'title')}")
print(f"Vector: {len(source_vec)} bytes ({len(source_vec)//4} × float32)\n")

# KNN 5 nearest neighbors
r_bin.execute_command('FT.SEARCH', 'idx:movies',
    '@vote_average:[-inf +inf]=>[KNN 5 @vector $query_vec]',
    'PARAMS', '2', 'query_vec', source_vec,
    'RETURN', '2', 'title', 'genres',
    'DIALECT', '2')

Source: Star Wars
Vector: 3072 bytes (768 × float32)



[5,
 b'movie:11',
 [b'title', b'Star Wars', b'genres', b'Adventure, Action, Science Fiction'],
 b'movie:1891',
 [b'title',
  b'The Empire Strikes Back',
  b'genres',
  b'Adventure, Action, Science Fiction'],
 b'movie:1892',
 [b'title',
  b'Return of the Jedi',
  b'genres',
  b'Adventure, Action, Science Fiction'],
 b'movie:1893',
 [b'title',
  b'Star Wars: Episode I - The Phantom Menace',
  b'genres',
  b'Adventure, Action, Science Fiction'],
 b'movie:1894',
 [b'title',
  b'Star Wars: Episode II - Attack of the Clones',
  b'genres',
  b'Adventure, Action, Science Fiction']]

### 2.6 Hybrid Search (Filter + Vector)

Combine tag/numeric filters with vector similarity — e.g., "Action movies similar to Star Wars".


In [17]:
# Hybrid: tag filter + vector KNN
# "Action movies similar to Star Wars"
r_bin.execute_command('FT.SEARCH', 'idx:movies',
    '@genres:{Action}=>[KNN 5 @vector $query_vec]',
    'PARAMS', '2', 'query_vec', source_vec,
    'RETURN', '3', 'title', 'genres', 'vote_average',
    'DIALECT', '2')

[5,
 b'movie:11',
 [b'title',
  b'Star Wars',
  b'genres',
  b'Adventure, Action, Science Fiction',
  b'vote_average',
  b'8.203'],
 b'movie:1891',
 [b'title',
  b'The Empire Strikes Back',
  b'genres',
  b'Adventure, Action, Science Fiction',
  b'vote_average',
  b'8.393'],
 b'movie:1892',
 [b'title',
  b'Return of the Jedi',
  b'genres',
  b'Adventure, Action, Science Fiction',
  b'vote_average',
  b'7.899'],
 b'movie:1893',
 [b'title',
  b'Star Wars: Episode I - The Phantom Menace',
  b'genres',
  b'Adventure, Action, Science Fiction',
  b'vote_average',
  b'6.6'],
 b'movie:1894',
 [b'title',
  b'Star Wars: Episode II - Attack of the Clones',
  b'genres',
  b'Adventure, Action, Science Fiction',
  b'vote_average',
  b'6.569']]

## Part 3: Single-Slot Index — Per-User History (~30 min)

A **single-slot index** lives entirely on ONE shard.
- All keys share the same hash slot (via key prefix)
- Queries execute locally — no fanout, no merge
- Microsecond latency (vs milliseconds for global)
- One index per user — ephemeral, created on login, dropped on logout

### 3.1 Load & Create Per-User Indexes


In [18]:
# Example: what one document looks like in a user's watch history
# Each rating becomes a HASH key: user:<id>:watch:<n>
print('''
Key:    user:1:watch:0
Fields:
  title     = "Toy Story"           (TEXT — searchable)
  genres    = "Adventure,Animation" (TAG — filterable)
  rating    = 4.0                   (NUMERIC — user's rating, 0.5-5.0)
  timestamp = 944249077             (NUMERIC — when they rated it)
  tmdb_id   = "862"                 (TAG — links to movie:862 in global catalog)
''')



Key:    user:1:watch:0
Fields:
  title     = "Toy Story"           (TEXT — searchable)
  genres    = "Adventure,Animation" (TAG — filterable)
  rating    = 4.0                   (NUMERIC — user's rating, 0.5-5.0)
  timestamp = 944249077             (NUMERIC — when they rated it)
  tmdb_id   = "862"                 (TAG — links to movie:862 in global catalog)



In [19]:
# Load user data
movies_df = pd.read_csv('data/movies.csv')
ratings_df = pd.read_csv('data/ratings.csv')

# Pick 3 users for the demo
demo_users = [int(x) for x in open('data/users.txt').read().split()[:3]]
print(f"Demo users: {demo_users}")

for user_id in demo_users:
    user_ratings = ratings_df[ratings_df['userId'] == user_id].merge(movies_df, on='movieId')
    user_ratings['genres'] = user_ratings['genres'].str.replace('|', ',')
    
    idx_name = f"idx:user:{user_id}:history"
    prefix = f"user:{user_id}:watch:"
    
    # Load data FIRST
    pipe = r.pipeline(transaction=False)
    for i, (_, row) in enumerate(user_ratings.iterrows()):
        tmdb_id = str(int(row['tmdbId'])) if pd.notna(row.get('tmdbId')) else ''
        pipe.hset(f"{prefix}{i}", mapping={
            'title': str(row['title']),
            'genres': str(row['genres']),
            'rating': str(row['rating']),
            'timestamp': str(int(row['timestamp'])),
            'tmdb_id': tmdb_id,
        })
    pipe.execute()
    
    # THEN create index (backfill async)
    try:
        r.execute_command('FT.DROPINDEX', idx_name)
    except:
        pass
    r.execute_command('FT.CREATE', idx_name, 'ON', 'HASH',
        'PREFIX', '1', prefix,
        'SCHEMA',
        'title', 'TEXT',
        'genres', 'TAG', 'SEPARATOR', ',',
        'rating', 'NUMERIC', 'SORTABLE',
        'timestamp', 'NUMERIC', 'SORTABLE',
        'tmdb_id', 'TAG')
    
    print(f"  User {user_id}: {len(user_ratings)} movies loaded + indexed")

print("\n✓ Single-slot indexes created")

Demo users: [1, 2, 3]
  User 1: 141 movies loaded + indexed
  User 2: 52 movies loaded + indexed
  User 3: 147 movies loaded + indexed

✓ Single-slot indexes created


In [20]:
# Inspect one user's index
r.execute_command('FT.INFO', f'idx:user:{demo_users[0]}:history')


['index_name',
 'idx:user:1:history',
 'index_definition',
 ['key_type', 'HASH', 'prefixes', ['user:1:watch:'], 'default_score', '1'],
 'attributes',
 [['identifier',
   'genres',
   'attribute',
   'genres',
   'user_indexed_memory',
   0,
   'type',
   'TAG',
   'SEPARATOR',
   ',',
   'CASESENSITIVE',
   '0',
   'size',
   '0'],
  ['identifier',
   'timestamp',
   'attribute',
   'timestamp',
   'user_indexed_memory',
   0,
   'type',
   'NUMERIC',
   'size',
   '0'],
  ['identifier',
   'rating',
   'attribute',
   'rating',
   'user_indexed_memory',
   0,
   'type',
   'NUMERIC',
   'size',
   '0'],
  ['identifier',
   'title',
   'attribute',
   'title',
   'user_indexed_memory',
   0,
   'type',
   'TEXT',
   'WITH_SUFFIX_TRIE',
   '0',
   'NO_STEM',
   '0',
   'WEIGHT',
   '1'],
  ['identifier',
   'tmdb_id',
   'attribute',
   'tmdb_id',
   'user_indexed_memory',
   0,
   'type',
   'TAG',
   'SEPARATOR',
   ',',
   'CASESENSITIVE',
   '0',
   'size',
   '0']],
 'num_docs',
 0

### 3.2 Query User History

In [21]:
user_id = demo_users[0]
idx = f"idx:user:{user_id}:history"

# Recent watches sorted by time
r.execute_command('FT.SEARCH', idx, '@timestamp:[-inf +inf]',
    'SORTBY', 'timestamp', 'DESC',
    'RETURN', '3', 'title', 'rating', 'timestamp',
    'LIMIT', '0', '5')

[0]

In [22]:
# Top-rated Action movies for this user
r.execute_command('FT.SEARCH', idx,
    '@genres:{Action} @rating:[4 5]',
    'SORTBY', 'rating', 'DESC',
    'RETURN', '2', 'title', 'rating',
    'LIMIT', '0', '5')

[0]

In [23]:
# Text search user's history: "Have I watched Star Wars?"
r.execute_command('FT.SEARCH', idx, 'Star Wars',
    'RETURN', '2', 'title', 'rating',
    'LIMIT', '0', '5')

[0]

## Part 4: Global User Index & FT.AGGREGATE (~30 min)

A **second global index** over ALL users' watch history.
- Distributed across shards (keys have no hash tag)
- Enables cross-user analytics: trending, most watched, avg ratings
- Queried with `FT.AGGREGATE` — server-side GROUPBY, REDUCE, SORT
- Single-slot answers "what did *I* watch?" — this answers "what is *everyone* watching?"

### 4.1 Load & Create


In [24]:
# Example: what one document looks like in the global watch index
# Each rating becomes a HASH key: watch:<userId>:<tmdbId>
print('''
Key:    watch:1:862
Fields:
  user_id   = "1"                   (TAG — which user)
  tmdb_id   = "862"                 (TAG — which movie, links to catalog)
  title     = "Toy Story"           (TEXT — searchable)
  genres    = "Adventure,Animation" (TAG — for genre analytics)
  rating    = 4.0                   (NUMERIC — for AVG/SUM aggregations)
  timestamp = 944249077             (NUMERIC — for time-based trending)
''')



Key:    watch:1:862
Fields:
  user_id   = "1"                   (TAG — which user)
  tmdb_id   = "862"                 (TAG — which movie, links to catalog)
  title     = "Toy Story"           (TEXT — searchable)
  genres    = "Adventure,Animation" (TAG — for genre analytics)
  rating    = 4.0                   (NUMERIC — for AVG/SUM aggregations)
  timestamp = 944249077             (NUMERIC — for time-based trending)



In [25]:
# Load ALL users' ratings FIRST
all_ratings = ratings_df.merge(movies_df, on='movieId')
all_ratings['genres'] = all_ratings['genres'].str.replace('|', ',')

pipe = r.pipeline(transaction=False)
count = 0
for _, row in all_ratings.iterrows():
    uid = str(int(row['userId']))
    tid = str(int(row['tmdbId'])) if pd.notna(row.get('tmdbId')) else ''
    if not tid:
        continue
    pipe.hset(f"watch:{uid}:{tid}", mapping={
        'user_id': uid,
        'tmdb_id': tid,
        'title': str(row['title']),
        'genres': str(row['genres']),
        'rating': str(row['rating']),
        'timestamp': str(int(row['timestamp'])),
    })
    count += 1
    if count % 500 == 0:
        pipe.execute()
        pipe = r.pipeline(transaction=False)
pipe.execute()
print(f"✓ Loaded {count} watch events")

# THEN create index (backfill async)
try:
    r.execute_command('FT.DROPINDEX', 'idx:watch')
except:
    pass

r.execute_command('FT.CREATE', 'idx:watch', 'ON', 'HASH', 'PREFIX', '1', 'watch:',
    'SCHEMA',
    'user_id', 'TAG',
    'tmdb_id', 'TAG',
    'title', 'TEXT',
    'genres', 'TAG', 'SEPARATOR', ',',
    'rating', 'NUMERIC', 'SORTABLE',
    'timestamp', 'NUMERIC', 'SORTABLE')

# Wait for backfill
import time
while True:
    info = r.execute_command('FT.INFO', 'idx:watch')
    d = dict(zip(info[::2], info[1::2]))
    if float(d.get('backfill_complete_percent', 1.0)) >= 1.0:
        break
    time.sleep(0.5)
print(f"✓ idx:watch ready: {d.get('num_docs', 0)} docs indexed")

✓ Loaded 1958 watch events
✓ idx:watch ready: 1958 docs indexed


In [26]:
# Inspect the global user index
r.execute_command('FT.INFO', 'idx:watch')


['index_name',
 'idx:watch',
 'index_definition',
 ['key_type', 'HASH', 'prefixes', ['watch:'], 'default_score', '1'],
 'attributes',
 [['identifier',
   'tmdb_id',
   'attribute',
   'tmdb_id',
   'user_indexed_memory',
   7453,
   'type',
   'TAG',
   'SEPARATOR',
   ',',
   'CASESENSITIVE',
   '0',
   'size',
   '1958'],
  ['identifier',
   'genres',
   'attribute',
   'genres',
   'user_indexed_memory',
   37768,
   'type',
   'TAG',
   'SEPARATOR',
   ',',
   'CASESENSITIVE',
   '0',
   'size',
   '1958'],
  ['identifier',
   'title',
   'attribute',
   'title',
   'user_indexed_memory',
   48426,
   'type',
   'TEXT',
   'WITH_SUFFIX_TRIE',
   '0',
   'NO_STEM',
   '0',
   'WEIGHT',
   '1'],
  ['identifier',
   'user_id',
   'attribute',
   'user_id',
   'user_indexed_memory',
   3518,
   'type',
   'TAG',
   'SEPARATOR',
   ',',
   'CASESENSITIVE',
   '0',
   'size',
   '1958'],
  ['identifier',
   'rating',
   'attribute',
   'rating',
   'user_indexed_memory',
   5874,
   'typ

### 4.2 FT.AGGREGATE — Platform Analytics

In [27]:
# FT.AGGREGATE: Most watched movies across all users
results = r.execute_command('FT.AGGREGATE', 'idx:watch', '@rating:[-inf +inf]',
    'LOAD', '1', '@tmdb_id',
    'GROUPBY', '1', '@tmdb_id',
    'REDUCE', 'COUNT', '0', 'AS', 'watch_count',
    'SORTBY', '2', '@watch_count', 'DESC',
    'LIMIT', '0', '10')

# Raw response
print("Raw:", results[:3], "...\n")

# Server computed the counts — we just look up titles
print("Most Watched:")
for row in results[1:]:
    doc = dict(zip(row[::2], row[1::2]))
    title = r.hget(f"movie:{doc['tmdb_id']}", 'title') or doc['tmdb_id']
    print(f"  {title} — {doc['watch_count']} watches")

Raw: [10, ['tmdb_id', '278', 'watch_count', '14'], ['tmdb_id', '680', 'watch_count', '12']] ...

Most Watched:
  The Shawshank Redemption — 14 watches
  Pulp Fiction — 12 watches
  Star Wars — 10 watches
  Schindler's List — 10 watches
  The Silence of the Lambs — 10 watches
  Forrest Gump — 9 watches
  Braveheart — 9 watches
  Back to the Future — 9 watches
  The Empire Strikes Back — 8 watches
  Fargo — 8 watches


In [28]:
# Highest rated movies (server-side AVG + COUNT + FILTER)
results = r.execute_command('FT.AGGREGATE', 'idx:watch', '@rating:[-inf +inf]',
    'LOAD', '2', '@tmdb_id', '@rating',
    'GROUPBY', '1', '@tmdb_id',
    'REDUCE', 'AVG', '1', '@rating', 'AS', 'avg_rating',
    'REDUCE', 'COUNT', '0', 'AS', 'num_ratings',
    'FILTER', '@num_ratings >= 3',
    'SORTBY', '2', '@avg_rating', 'DESC',
    'LIMIT', '0', '10')

for row in results[1:]:
    doc = dict(zip(row[::2], row[1::2]))
    title = r.hget(f"movie:{doc['tmdb_id']}", 'title') or doc['tmdb_id']
    print(f"  {title} — avg {float(doc['avg_rating']):.2f}★ ({doc['num_ratings']} ratings)")

  Citizen Kane — avg 5.00★ (3 ratings)
  The Graduate — avg 5.00★ (3 ratings)
  Traffic — avg 4.83★ (3 ratings)
  North by Northwest — avg 4.75★ (4 ratings)
  Breakfast at Tiffany's — avg 4.67★ (3 ratings)
  2001: A Space Odyssey — avg 4.67★ (3 ratings)
  Secrets & Lies — avg 4.67★ (3 ratings)
  Das Boot — avg 4.67★ (3 ratings)
  The Breakfast Club — avg 4.62★ (4 ratings)
  A Fish Called Wanda — avg 4.62★ (4 ratings)


In [29]:
# Most popular genres (GROUPBY on TAG field)
results = r.execute_command('FT.AGGREGATE', 'idx:watch', '@rating:[-inf +inf]',
    'LOAD', '1', '@genres',
    'GROUPBY', '1', '@genres',
    'REDUCE', 'COUNT', '0', 'AS', 'watch_count',
    'SORTBY', '2', '@watch_count', 'DESC',
    'LIMIT', '0', '10')

for row in results[1:]:
    doc = dict(zip(row[::2], row[1::2]))
    print(f"  {doc['genres']} — {doc['watch_count']} watches")

  Drama — 174 watches
  Drama,Romance — 87 watches
  Comedy — 86 watches
  Comedy,Romance — 73 watches
  Comedy,Drama,Romance — 72 watches
  Comedy,Drama — 72 watches
  Crime,Drama — 66 watches
  Action,Adventure,Sci-Fi — 58 watches
  Action,Adventure,Sci-Fi,Thriller — 38 watches
  Drama,Thriller — 33 watches


## Part 5: End-to-End Recommendation Flow (~30 min)

This is the payoff — combining all three indexes to build personalized recommendations.

**Flow:**
1. Query single-slot → user's top genre + favorite movie (μs)
2. Fetch that movie's vector from global catalog (key lookup)
3. KNN on global catalog → similar movies in that genre (ms)

### 5.1 The Full Pipeline


In [30]:
user_id = demo_users[0]
idx = f"idx:user:{user_id}:history"

print(f"=== Personalized Recommendations for User {user_id} ===\n")

# Step 1: Find user's top genre — server-side with FT.AGGREGATE
print("Step 1: What genre does this user watch most?")
genre_results = r.execute_command('FT.AGGREGATE', idx, '@rating:[-inf +inf]',
    'LOAD', '1', '@genres',
    'GROUPBY', '1', '@genres',
    'REDUCE', 'COUNT', '0', 'AS', 'count',
    'SORTBY', '2', '@count', 'DESC',
    'LIMIT', '0', '1')

# genres field is comma-separated — pick the first genre from the top group
top_genres_str = genre_results[1][1]  # e.g. 'Drama' or 'Comedy,Drama'
top_genre = top_genres_str.split(',')[0]
print(f"  → Top genre group: {top_genres_str} (count: {genre_results[1][3]})")
print(f"  → Using: {top_genre}\n")

# Step 2: Get their most recent highly-rated movie in that genre
print(f"Step 2: Recent top-rated {top_genre} movie?")
recent = r.execute_command('FT.SEARCH', idx,
    f'@genres:{{{top_genre}}} @rating:[4 5]',
    'SORTBY', 'timestamp', 'DESC',
    'RETURN', '3', 'title', 'rating', 'tmdb_id',
    'LIMIT', '0', '1')
if recent[0] > 0:
    seed = dict(zip(recent[2][::2], recent[2][1::2]))
    print(f"  → '{seed['title']}' rated {seed['rating']}★ (tmdb_id: {seed['tmdb_id']})\n")
else:
    print("  → No match found")
    seed = None

=== Personalized Recommendations for User 1 ===

Step 1: What genre does this user watch most?
  → Top genre group: Drama (count: 24)
  → Using: Drama

Step 2: Recent top-rated Drama movie?
  → 'Citizen Ruth (1996)' rated 4.0★ (tmdb_id: 13891)



In [31]:
# Step 3: Fetch vector from global catalog
if seed and seed.get('tmdb_id'):
    tmdb_id = seed['tmdb_id']
    title = seed['title']
    print(f'Step 3: Fetch vector for movie:{tmdb_id}')
    vec_blob = r_bin.hget(f'movie:{tmdb_id}'.encode(), b'vector')
    if vec_blob:
        print(f'  \u2192 {len(vec_blob)} bytes ({len(vec_blob)//4} floats)\n')

        # Step 4: KNN — find 3 similar movies in that genre
        print(f'Step 4: Find {top_genre} movies similar to \'{title}\'')
        knn = r_bin.execute_command('FT.SEARCH', 'idx:movies',
            f'@genres:{{{top_genre}}}=>[KNN 3 @vector $q]',
            'PARAMS', '2', 'q', vec_blob,
            'RETURN', '3', 'title', 'genres', 'vote_average',
            'DIALECT', '2')

        print('  Recommendations:')
        for i in range(1, len(knn), 2):
            d = dict(zip(knn[i+1][::2], knn[i+1][1::2]))
            print(f'    {d[b"title"].decode()} ({d[b"vote_average"].decode()}\u2605)')
    else:
        print(f'  \u2192 movie:{tmdb_id} not in catalog')
else:
    print('  \u2192 Skipped (no seed movie)')


Step 3: Fetch vector for movie:13891
  → 3072 bytes (768 floats)

Step 4: Find Drama movies similar to 'Citizen Ruth (1996)'
  Recommendations:
    Citizen Ruth (6.6★)
    Boys on the Side (6.3★)
    The People vs. Larry Flynt (7.0★)


## Part 6: Exercises (~30 min)

### Exercise 1: Multi-language search
Find French (`fr`) comedies rated above 7. Hint: use `@original_language:{fr}`.

### Exercise 2: User dedup
Before recommending a movie, check if the user already watched it using their single-slot index.

### Exercise 3: Trending + Personal
Combine FT.AGGREGATE (trending movies) with the user's single-slot (exclude already watched).

### Exercise 4: Custom aggregation
Write an FT.AGGREGATE query to find the average rating per genre across all users.


In [32]:
# Exercise 1: French comedies rated 7+
# YOUR CODE HERE
results = r.execute_command('FT.SEARCH', 'idx:movies',
    '@genres:{Comedy} @original_language:{fr} @vote_average:[7 10]',
    'RETURN', '3', 'title', 'vote_average', 'original_language',
    'LIMIT', '0', '5')
print(f"French comedies rated 7+: {results[0]}")
for i in range(1, len(results), 2):
    doc = dict(zip(results[i+1][::2], results[i+1][1::2]))
    print(f"  {doc['title']} — {doc['vote_average']}★")

French comedies rated 7+: 5
  Little White Lies — 7.112★
  Amélie — 7.9★
  The Intouchables — 8.3★
  Fear City: A Family-Style Comedy — 7.501★
  Three Colors: White — 7.46★


In [33]:
# Exercise 4: Average rating per genre
# YOUR CODE HERE
results = r.execute_command('FT.AGGREGATE', 'idx:watch', '@rating:[-inf +inf]',
    'LOAD', '2', '@genres', '@rating',
    'GROUPBY', '1', '@genres',
    'REDUCE', 'AVG', '1', '@rating', 'AS', 'avg_rating',
    'REDUCE', 'COUNT', '0', 'AS', 'count',
    'FILTER', '@count >= 10',
    'SORTBY', '2', '@avg_rating', 'DESC',
    'LIMIT', '0', '10')

print("Average rating per genre (min 10 ratings):")
for row in results[1:]:
    doc = dict(zip(row[::2], row[1::2]))
    print(f"  {doc['genres']} — avg {float(doc['avg_rating']):.2f}★ ({doc['count']} ratings)")

Average rating per genre (min 10 ratings):
  Documentary — avg 4.23★ (11 ratings)
  Crime,Mystery,Thriller — avg 4.21★ (14 ratings)
  Comedy,Crime,Drama,Thriller — avg 4.21★ (21 ratings)
  Action,Adventure — avg 4.21★ (12 ratings)
  Comedy,Crime — avg 4.15★ (17 ratings)
  Crime,Drama — avg 4.14★ (66 ratings)
  Drama,War — avg 4.13★ (30 ratings)
  Mystery,Thriller — avg 4.11★ (14 ratings)
  Action,Adventure,Drama — avg 4.08★ (13 ratings)
  Drama,Romance — avg 4.00★ (87 ratings)


## Summary

### What You Learned

| Concept | Command | Use Case |
|:--------|:--------|:---------|
| Full-text search | `FT.SEARCH idx "query"` | Find movies by keywords |
| Tag filter | `@field:{value}` | Filter by genre, language |
| Numeric range | `@field:[min max]` | Filter by rating, date |
| Sort | `SORTBY field DESC` | Order results |
| Vector KNN | `*=>[KNN k @vector $blob]` | Find similar items |
| Hybrid search | `@filter=>[KNN k @vector $blob]` | Filtered similarity |
| Aggregation | `FT.AGGREGATE ... GROUPBY ... REDUCE` | Analytics, trending |
| Single-slot | Per-user index, no fanout | Microsecond user queries |

### Three Index Pattern

```
Global Catalog (idx:movies)     → Browse, search, vector similarity
Single-Slot (idx:user:X:history) → Per-user, microsecond, ephemeral
Global Users (idx:watch)         → Cross-user analytics, trending
```

### Key Takeaway
Single-slot queries avoid fanout overhead, making them significantly faster for user-specific data.
Use global indexes for shared data (catalog) and cross-user analytics (trending).
